# Colab 37 - symbol statistics of the training generator and the three representations

**Why:** supervisor notes 33/34 reject the adjectives *diffuse* / *concentrated* in Section 3.4 and
ask for a number. This produces that number plus the appendix figures (note 32).

**What it does NOT do:** no model, no training, no evaluation. Descriptive statistics only.

**Design point:** the collections are rebuilt with the *same* filter as the run of record - same
`[50, 200]`, same two exempted domains - and the Synth series uses the *same generator* as
Section 3.3, so every series describes what the thesis actually specifies. Assertions stop the
notebook if the collection sizes drift from 10,501 / 10,497 / 10,501.

The palette is the locked deck palette - Synth orange, 3Di blue, SS red, AA dark grey - so these
figures match the slides and the rest of the thesis.

## 1. Setup - clone the repository and locate the data

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')
!pip install rapidfuzz --quiet


In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
for f in ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz', 'cath_s20_3di.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')


In [ ]:
import numpy as np, pandas as pd, json
import matplotlib.pyplot as plt
from collections import Counter
from rapidfuzz.distance import Levenshtein as RFLev

# --- IDENTICAL to the run of record. Do not change without changing the thesis. ---
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
SS_ALPHABET = 'HLS'
MIN_LEN, MAX_LEN = 50, 200
RESCUED   = {'4z0mC02', '3qkaE02'}   # kept per the 2026-08-24 decision; disclosed in Section 3.4
N_TRAIN   = 30_000                   # Section 3.3: training pairs per seed
TRAIN_SEED = 0
STRAT_CAND = 200_000                 # candidate pairs sampled per collection, run of record
PAIR_SEED  = 999                     # the run of record's pair-sampling seed

AA_SET, SS_SET = set(AA_ALPHABET), set(SS_ALPHABET)
is_aa = lambda s: all(c in AA_SET for c in s)
is_ss = lambda s: all(c in SS_SET for c in s)

def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - RFLev.distance(a, b) / L

# ---------------------------------------------------------------------------
# LOCKED DECK PALETTE (confirmed 2026-08-24), colour-picked from the slides:
#   synth orange / 3Di blue / SS red / AA dark grey.
# The earlier local figures used AA blue / 3Di purple / SS green - that palette
# is retired. Do not reintroduce it.
# ---------------------------------------------------------------------------
COLOUR = {'Synth': '#FF7F0E', '3Di': '#0072B2', 'SS': '#D62728', 'AA': '#4D4D4D'}
ORDER  = ['Synth', 'AA', '3Di', 'SS']          # symbol analyses: one Synth series

# The score plot separates the two synthetic sets. The locked palette has a single
# synth colour, so they share the orange hue and are told apart by line style.
COLOUR['Synth Train'] = '#FF7F0E'
COLOUR['Synth Eval']  = '#A85700'
COLOUR['Synth Eval (balanced)'] = '#A85700'   # same set, style tells them apart
SCORE_ORDER = ['Synth Train', 'Synth Eval', 'Synth Eval (balanced)', 'AA', 'SS', '3Di']
SYN_PERTURB, SYN_INDEP = 20_000, 8_000         # run of record
SYN_SEED, STRAT_PER_BIN = 20260810, 400


## 2. Rebuild the three collections (identical filter to the run of record)

In [ ]:
raw = pd.concat([pd.read_csv(f'{DATA_DIR}/cath_s20_train70.csv.gz'),
                 pd.read_csv(f'{DATA_DIR}/cath_s20_test30.csv.gz')],
                ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(f'{DATA_DIR}/cath_s20_3di.csv.gz')

def _valid(seq, isstd, d):
    return (isinstance(seq, str) and isstd(seq)
            and ((MIN_LEN <= len(seq) <= MAX_LEN) or d in RESCUED))

id_to_aa  = {d: s for d, s in zip(raw['domain_id'], raw['aa_seq'])              if _valid(s, is_aa, d)}
id_to_ss  = {d: s for d, s in zip(raw['domain_id'], raw['ss_seq'])              if _valid(s, is_ss, d)}
id_to_3di = {d: s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_aa, d)}

COLLECTION = {'AA': list(id_to_aa.values()),
              'SS': list(id_to_ss.values()),
              '3Di': list(id_to_3di.values())}

print('Collection sizes  (run of record: AA 10,501 / SS 10,497 / 3Di 10,501)')
for r in ['AA', 'SS', '3Di']:
    n = len(COLLECTION[r]); tot = sum(len(s) for s in COLLECTION[r])
    print(f'  {r:<4} sequences = {n:>6,}   symbols = {tot:>9,}')
assert len(COLLECTION['AA'])  == 10_501, 'AA collection differs from the run of record - STOP'
assert len(COLLECTION['SS'])  == 10_497, 'SS collection differs from the run of record - STOP'
assert len(COLLECTION['3Di']) == 10_501, '3Di collection differs from the run of record - STOP'
print('\nMatches the run of record.')


## 3. Regenerate the training set (identical generator to Section 3.3)

The Synth series is the seed-0 training set: both strings of all 30,000 pairs for the symbol
statistics, and the exact labels for the score distribution. The printed far/mid/high counts should
reproduce 5 / 19,013 / 10,982 - if they do not, the generator has drifted from the run of record.

In [ ]:
# --- the training generator, verbatim from the run of record (Section 3.3) ---
def rand_seq(abc, rng):
    L = int(rng.integers(MIN_LEN, MAX_LEN + 1))
    return ''.join(rng.choice(list(abc), size=L))

def perturb(seq, k, abc, rng):
    s = list(seq); abc = list(abc)
    for _ in range(k):
        if len(s) == 0:            op = 'ins'
        elif len(s) >= MAX_LEN:    op = rng.choice(['sub', 'del'])
        else:                      op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s)); s[i] = rng.choice([c for c in abc if c != s[i]])
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1); s.insert(i, rng.choice(abc))
        else:
            i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)

def build_pairs(n, seed):
    rng = np.random.default_rng(seed); pairs = []
    while len(pairs) < n:
        sd = rand_seq(AA_ALPHABET, rng); L = len(sd)
        t = float(rng.uniform(0, 1)); k = max(0, int(round((1 - t) * L)))
        o = perturb(sd, k, AA_ALPHABET, rng)
        if 1 <= len(o) <= MAX_LEN:
            pairs.append((sd, o, norm_lev(sd, o)))
    return pairs

print(f'generating the seed-{TRAIN_SEED} training set ({N_TRAIN:,} pairs) - about a minute...')
TRAIN_PAIRS = build_pairs(N_TRAIN, TRAIN_SEED)
TRAIN_LABELS = np.array([l for *_, l in TRAIN_PAIRS])

# the Synth series enters the symbol analyses as the strings of those pairs
COLLECTION['Synth'] = [s for a, b, _ in TRAIN_PAIRS for s in (a, b)]
ALPHABET = {'Synth': AA_ALPHABET, 'AA': AA_ALPHABET, '3Di': AA_ALPHABET, 'SS': SS_ALPHABET}

far = int((TRAIN_LABELS < 0.30).sum())
mid = int(((TRAIN_LABELS >= 0.30) & (TRAIN_LABELS < 0.70)).sum())
high = int((TRAIN_LABELS >= 0.70).sum())
print(f'  training labels  far/mid/high = {far} / {mid:,} / {high:,}   median {np.median(TRAIN_LABELS):.3f}')
print(f'  (Section 3.3 records 5 / 19,013 / 10,982 at seed 0)')
print(f'  Synth strings = {len(COLLECTION["Synth"]):,}, '
      f'symbols = {sum(len(s) for s in COLLECTION["Synth"]):,}')


## 4. Rebuild the Synth evaluation set (identical generator to the run of record)

Disjoint from the training set and drawn from its own stream. Two things it does differently, both
verbatim from the run of record: the edit count is drawn uniformly over $\{0,\dots,L\}$ rather than
through $t$, and 8,000 pairs of independently generated strings are added to populate the low range.
The collection is then the strings of the decile-balanced pairs, which is why it is 7,296 sequences
rather than the full generated set - unlike the CATH collections, which are the complete filtered
domain sets.

In [ ]:
# --- the synthetic EVALUATION set, verbatim from the run of record ---
# Note it differs from training in how the edit count is drawn: uniformly over
# {0,...,L} here, versus k = round((1-t)L) with t ~ U(0,1) in Section 3.3.
def build_synth_feed(n_perturb, n_indep, per_bin=STRAT_PER_BIN, seed=SYN_SEED):
    r = np.random.default_rng(seed); recs = []
    for _ in range(n_perturb):
        base = rand_seq(AA_ALPHABET, r)
        part = perturb(base, int(r.integers(0, len(base) + 1)), AA_ALPHABET, r)
        if 1 <= len(part) <= MAX_LEN:
            recs.append((base, part))
    for _ in range(n_indep):
        recs.append((rand_seq(AA_ALPHABET, r), rand_seq(AA_ALPHABET, r)))
    recs = [(a, b, norm_lev(a, b)) for a, b in recs]
    nl_all = np.array([x[2] for x in recs])          # BEFORE decile balancing
    bins = np.clip(np.digitize(nl_all, np.linspace(0, 1, 11)) - 1, 0, 9); take = []
    for bb in range(10):
        idx = np.where(bins == bb)[0]
        if idx.size:
            take.extend(r.permutation(idx)[:per_bin].tolist())
    seqs, NL = [], []
    for idx in take:
        a, b, l = recs[int(idx)]
        seqs.append(a); seqs.append(b); NL.append(l)
    return seqs, np.array(NL), nl_all

print(f'generating the Synth evaluation set ({SYN_PERTURB:,} perturbation + '
      f'{SYN_INDEP:,} independent pairs) - about a minute...')
SYN_SEQ, SYN_NL_BALANCED, SYN_NL_ALL = build_synth_feed(SYN_PERTURB, SYN_INDEP)

print(f'  generated pairs          = {len(SYN_NL_ALL):,}   median {np.median(SYN_NL_ALL):.3f}')
print(f'  after decile balancing   = {len(SYN_NL_BALANCED):,} pairs '
      f'(median {np.median(SYN_NL_BALANCED):.3f} - flat by construction)')
print(f'  Synth Eval collection    = {len(SYN_SEQ):,} sequences '
      f'(run of record: 7,296)')
assert len(SYN_SEQ) == 7_296, 'Synth Eval collection differs from the run of record - STOP'
print('\nMatches the run of record.')


## 5. Letter-frequency profiles and entropy

`normalised_entropy` is $H/\log_2 k$: 1.0 means every symbol equally likely, lower means the mass
concentrates on fewer symbols. This is the number that replaces "diffuse" and "concentrated", and the
only one comparable across all four series - `top5_share` is 1.0 for SS by construction, since SS has
a three-letter alphabet.

In [ ]:
def profile(seqs, alphabet):
    obs = Counter()
    for s in seqs:
        obs.update(s)
    counts = pd.Series({c: obs.get(c, 0) for c in alphabet}, dtype=np.int64)
    unknown = {c: n for c, n in obs.items() if c not in set(alphabet)}
    if unknown:
        print(f'  ! symbols outside the declared alphabet, not counted: {unknown}')
    return counts, counts / counts.sum()

def entropy_bits(p):
    q = np.asarray(p, float); q = q[q > 0]
    return float(-(q * np.log2(q)).sum())

rows, PROF = [], {}
for r in ORDER:
    counts, p = profile(COLLECTION[r], ALPHABET[r])
    PROF[r] = p
    H = entropy_bits(p); k = len(ALPHABET[r])
    rows.append(dict(series=r, alphabet_size=k, sequences=len(COLLECTION[r]),
                     symbols=int(counts.sum()),
                     entropy_bits=round(H, 3),
                     max_entropy_bits=round(np.log2(k), 3),
                     normalised_entropy=round(H / np.log2(k), 3),
                     effective_alphabet=round(2 ** H, 2),
                     top5_share=round(float(p.sort_values(ascending=False).head(5).sum()), 3),
                     most_frequent=p.idxmax(),
                     most_frequent_share=round(float(p.max()), 3)))

SYMSTATS = pd.DataFrame(rows)
SYMSTATS


## 6. First-order transition probabilities

The conditional entropy $H(X_{t+1} \mid X_t)$ is how much uncertainty about the next symbol survives
knowing the current one. The difference from the first-order entropy is the mutual information: how
much structure lives in **order** rather than in composition. The Synth series is the control -
it is generated i.i.d., so its mutual information should sit at ~0.

In [ ]:
def transition(seqs, alphabet):
    idx = {c: i for i, c in enumerate(alphabet)}; k = len(alphabet)
    flat = np.zeros(k * k, dtype=np.int64)
    for s in seqs:
        a = np.fromiter((idx[c] for c in s if c in idx), dtype=np.int64)
        if a.size > 1:
            flat += np.bincount(a[:-1] * k + a[1:], minlength=k * k)
    M = flat.reshape(k, k)
    P = M / np.maximum(M.sum(1, keepdims=True), 1)
    return M, pd.DataFrame(P, index=list(alphabet), columns=list(alphabet))

TRANSITION, cond_rows = {}, []
for r in ORDER:
    M, P = transition(COLLECTION[r], ALPHABET[r])
    TRANSITION[r] = P
    w = M.sum(1) / M.sum()
    Hc = float(sum(w[i] * entropy_bits(P.iloc[i]) for i in range(len(w))))
    H1 = float(SYMSTATS.loc[SYMSTATS['series'] == r, 'entropy_bits'].iloc[0])
    cond_rows.append(dict(series=r, entropy_bits=round(H1, 3),
                          conditional_entropy_bits=round(Hc, 3),
                          mutual_information_bits=round(H1 - Hc, 3)))

COND = pd.DataFrame(cond_rows)
print('How much of the structure lives in ORDER rather than composition:')
COND


## 7. Score distribution

The three CATH series are normalised Levenshtein similarity over randomly sampled pairs of each
collection - the **natural** pair distribution, not the decile-balanced evaluation set of Section 3.6.
Same candidate count (200,000) and seed (999) as the run of record.

**Synth Train** is the training set's own labels. **Synth Eval** is the generated pairs *before*
decile balancing; **Synth Eval (balanced)** is the evaluation set actually used, after balancing.
Together they show what the protocol of Section 3.6 does to a distribution.

The three CATH sets are shown unbalanced only. Reproducing their balanced sets faithfully needs the
exact-relevance-set scan, which injects every known high-similarity pair before balancing - without
it the AA high range would be empty, since AA has five such pairs in total. That scan is the
expensive part of the run of record and is deliberately not repeated here.

In [ ]:
# Score distribution. The CATH series are normLev over randomly sampled pairs of the
# collection - the natural distribution, NOT the decile-balanced evaluation set.
# Same candidate count and seed as the run of record.
def sampled_scores(seqs, n_cand, seed):
    rng = np.random.default_rng(seed); N = len(seqs)
    a = rng.integers(0, N, n_cand); b = rng.integers(0, N, n_cand)
    keep = a != b; a, b = a[keep], b[keep]
    return np.array([norm_lev(seqs[i], seqs[j]) for i, j in zip(a, b)])

SCORES = {'Synth Train': TRAIN_LABELS,
          'Synth Eval': SYN_NL_ALL,
          'Synth Eval (balanced)': SYN_NL_BALANCED}
for r in ['AA', '3Di', 'SS']:
    print(f'sampling {STRAT_CAND:,} pairs for {r}...')
    SCORES[r] = sampled_scores(COLLECTION[r], STRAT_CAND, PAIR_SEED)

MEDIAN = {r: float(np.median(SCORES[r])) for r in SCORE_ORDER}
print('\nmedians:', {r: round(v, 3) for r, v in MEDIAN.items()})
print('(the local score-distribution script carries AA 0.198 / SS 0.438 / 3Di 0.237)')


## 8. Figures

In [ ]:
# --- letter-frequency profiles, each alphabet sorted most -> least frequent ---
plt.figure(figsize=(10, 4.2))
for r in ORDER:
    p = PROF[r].sort_values(ascending=False)
    x = np.arange(1, len(p) + 1)
    plt.plot(x, p.values, 'o-', ms=4, lw=1.8, color=COLOUR[r],
             label=f'{r}  ({len(ALPHABET[r])} letters)')
    for i in range(min(3, len(p))):          # annotate the leading letters
        plt.annotate(p.index[i], (x[i], p.values[i]), textcoords='offset points',
                     xytext=(0, 7), ha='center', fontsize=9, color=COLOUR[r])
plt.axhline(1 / 20, ls='--', lw=1, color='0.45', alpha=0.9)
plt.text(20.1, 1 / 20, 'uniform (1/20)', fontsize=8, color='0.45', va='center')
plt.xticks(range(1, 21)); plt.xlim(0.5, 21.5)
plt.xlabel('letter rank within alphabet  (1 = most frequent)'); plt.ylabel('frequency')
plt.title('Letter-frequency profiles, each alphabet sorted most$\\rightarrow$least frequent')
plt.legend(frameon=False)
plt.gca().spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('colab37_letter_frequency.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# --- transition heatmaps, axes sorted by letter frequency, shared colour bar ---
fig, axes = plt.subplots(1, 4, figsize=(17, 4.4))
for ax, r in zip(axes, ORDER):
    order_letters = list(PROF[r].sort_values(ascending=False).index)
    P = TRANSITION[r].loc[order_letters, order_letters]
    im = ax.imshow(P.values, cmap='Reds', vmin=0, vmax=1)
    ax.set_title(r, color=COLOUR[r], fontweight='bold')
    ax.set_xticks(range(len(P))); ax.set_xticklabels(P.columns, fontsize=6)
    ax.set_yticks(range(len(P))); ax.set_yticklabels(P.index, fontsize=6)
    ax.set_xlabel('next letter (freq-sorted)')
    if r == ORDER[0]:
        ax.set_ylabel('current letter (freq-sorted)')
fig.suptitle('Transition probability P(next | current) - rows/cols sorted by letter '
             'frequency (most$\\rightarrow$least). SS collapses onto its 3-letter '
             'self-transition block.', fontsize=9, y=1.02)
fig.colorbar(im, ax=axes, fraction=0.012, pad=0.01, label='P(next | current)')
plt.savefig('colab37_transitions.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# --- score distribution: training set vs the three evaluation alphabets ---
EDGES = np.linspace(0, 1, 81); CENTERS = (EDGES[:-1] + EDGES[1:]) / 2
_smooth = lambda y, k=3: np.convolve(np.asarray(y, float), np.ones(k) / k, mode='same')

plt.figure(figsize=(9, 5))
for r in SCORE_ORDER:
    dens, _ = np.histogram(SCORES[r], bins=EDGES, density=True)
    y = _smooth(dens)
    ls = {'Synth Train': '--', 'Synth Eval': ':', 'Synth Eval (balanced)': '-'}.get(r, '-')
    plt.plot(CENTERS, y, ls, lw=2.2, color=COLOUR[r],
             label=f'{r}  (median {MEDIAN[r]:.2f})')
    plt.fill_between(CENTERS, y, color=COLOUR[r], alpha=0.10)
plt.axvline(0.70, ls='--', lw=1, color='0.4')
plt.text(0.71, plt.ylim()[1] * 0.92, 'high-sim bar (0.70)', fontsize=8, color='0.4')
plt.xlabel('normalised Levenshtein similarity (true score)'); plt.ylabel('density')
plt.title('Score distribution - the two synthetic sets vs the three evaluation alphabets')
plt.legend(title='set', frameon=False); plt.xlim(0, 1)
plt.gca().spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('colab37_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Save everything

In [ ]:
SYMSTATS.to_csv('colab37_symbol_statistics.csv', index=False)
COND.to_csv('colab37_conditional_entropy.csv', index=False)
for r in ORDER:
    TRANSITION[r].to_csv(f'colab37_transitions_{r}.csv')

summary = dict(collections={r: len(COLLECTION[r]) for r in ORDER},
               synth_eval_collection=len(SYN_SEQ),
               medians={r: round(MEDIAN[r], 4) for r in SCORE_ORDER},
               symbol_statistics=SYMSTATS.to_dict('records'),
               conditional_entropy=COND.to_dict('records'))
with open('colab37_summary.json', 'w') as fh:
    json.dump(summary, fh, indent=2)
print(json.dumps(summary, indent=2))

print('\n--- numbers for Section 3.4 ---')
for _, x in SYMSTATS.iterrows():
    print(f'  {x["series"]:<10} H = {x.entropy_bits} of {x.max_entropy_bits} bits '
          f'({x.normalised_entropy} normalised), effective alphabet {x.effective_alphabet}, '
          f'most frequent {x.most_frequent} at {x.most_frequent_share}')
print('\nDownload colab37_summary.json and the three PNGs.')
